# Matrix Factorization for Recommendation

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

from helpers.data_loaders import load_movielens_data, load_steam_data

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Using device: cuda


## 2. Data Loading and Preparation

In [ ]:
movies_df, ratings_df = load_movielens_data()
reviews_df_steam, items_df_steam = load_steam_data()

movielens_interactions = pd.DataFrame({
    'user_id': 'movielens_user_' + ratings_df['userId'].astype(str),
    'item_id': 'movielens_item_' + ratings_df['movieId'].astype(str),
    'rating': ratings_df['rating']
})


reviews_df_steam_filtered = reviews_df_steam.copy()
steam_interactions = pd.DataFrame({
    'user_id': 'steam_user_' + reviews_df_steam_filtered['user_id'].astype(str),
    'item_id': 'steam_item_' + reviews_df_steam_filtered['app_id'].astype(str),
    'rating': 1.0 # Implicit rating
})

all_interactions = pd.concat([movielens_interactions, steam_interactions]).drop_duplicates()

print(f"Total unique interactions: {len(all_interactions)}")

unique_users = all_interactions['user_id'].unique()
unique_items = all_interactions['item_id'].unique()

user_map = {user: i for i, user in enumerate(unique_users)}
item_map = {item: i for i, item in enumerate(unique_items)}

num_users = len(user_map)
num_items = len(item_map)

print(f"Number of users: {num_users}")
print(f"Number of items: {num_items}")

all_interactions['user_idx'] = all_interactions['user_id'].map(user_map)
all_interactions['item_idx'] = all_interactions['item_id'].map(item_map)


Total unique interactions: 25044685
Number of users: 184618
Number of items: 61849


## 3. Matrix Factorization Model Definition

In [3]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)
        
    def forward(self, user_indices, item_indices):
        user_emb = self.user_embedding(user_indices)
        item_emb = self.item_embedding(item_indices)
        
        # Dot product of user and item embeddings
        rating = torch.sum(user_emb * item_emb, dim=1)
        
        return rating


## 4. Training Setup

In [4]:
class RatingDataset(Dataset):
    def __init__(self, interactions):
        self.interactions = interactions

    def __len__(self):
        return len(self.interactions)

    def __getitem__(self, idx):
        interaction = self.interactions.iloc[idx]
        return (
            interaction['user_idx'],
            interaction['item_idx'],
            interaction['rating']
        )

embedding_dim = 32
batch_size = 4096
learning_rate = 1e-3
epochs = 1
lambda_reg = 1e-5

train_dataset = RatingDataset(all_interactions)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model = MatrixFactorization(num_users, num_items, embedding_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()


## 5. Training Loop

In [5]:
model.train()
for epoch in range(epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for user_batch, item_batch, rating_batch in progress_bar:
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        rating_batch = rating_batch.float().to(device)
        
        optimizer.zero_grad()
        
        predictions = model(user_batch, item_batch)
        
        # MSE Loss
        mse_loss = loss_fn(predictions, rating_batch)
        
        # L2 Regularization
        user_emb_reg = model.user_embedding(user_batch).norm(2).pow(2)
        item_emb_reg = model.item_embedding(item_batch).norm(2).pow(2)
        reg_loss = lambda_reg * (user_emb_reg + item_emb_reg) / len(user_batch)
        
        loss = mse_loss + reg_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}")


Epoch 1/1:   0%|          | 0/6115 [00:00<?, ?it/s]

Epoch 1/1, Average Loss: 3.7818


## 6. Making Recommendations

In [ ]:
def get_recommendations_mf(user_id_str, model, top_k=10):
    model.eval()
    
    if user_id_str not in user_map:
        print(f"User '{user_id_str}' not found.")
        return
    user_idx = user_map[user_id_str]
    
    with torch.no_grad():
        user_idx_tensor = torch.LongTensor([user_idx]).to(device)
        item_indices_tensor = torch.LongTensor(list(item_map.values())).to(device)
        
        # Predict ratings for all items for the given user
        scores = model(user_idx_tensor.repeat(num_items), item_indices_tensor)
        
        # Remove items the user has already interacted with
        liked_items_indices = all_interactions[all_interactions['user_id'] == user_id_str]['item_idx'].values
        scores[liked_items_indices] = -np.inf
        
        top_k_scores, top_k_indices = torch.topk(scores, k=top_k)
        
        inv_item_map = {i: item for item, i in item_map.items()}
        
        print(f"Top {top_k} recommendations for user '{user_id_str}':")
        for i, score in zip(top_k_indices.cpu().numpy(), top_k_scores.cpu().numpy()):
            item_id_str = inv_item_map[i]
            if 'steam' in item_id_str:
                item_info = items_df_steam.loc[items_df_steam['app_id'] == int(item_id_str.split('_')[-1])]
                if not item_info.empty:
                    print(f"  - [steam] Item: {item_id_str}, Title: {item_info['title'].values[0]} , Score: {score:.4f}")
            else:
                item_info = movies_df.loc[movies_df['movieId'] == int(item_id_str.split('_')[-1])]
                if not item_info.empty:
                    print(f"  - [movie] Item: {item_id_str}, Title: {item_info['title'].values[0]} , Score: {score:.4f}")

sample_user_id = 'steam_user_LydiaMorley'
print(f"Items liked by the user ({sample_user_id}):")

liked_items = all_interactions[all_interactions['user_id'] == sample_user_id]['item_id']
for item in liked_items:
    if 'steam' in item:
        item_info = items_df_steam.loc[items_df_steam['app_id'] == int(item.split('_')[-1])]
        if not item_info.empty:
            print(f"  - {item}, Title: {item_info['title'].values[0]}")
    else:
        item_info = movies_df.loc[movies_df['movieId'] == int(item.split('_')[-1])]
        if not item_info.empty:
            print(f"  - {item}, Title: {item_info['title'].values[0]}")

get_recommendations_mf(sample_user_id, model)


Items liked by the user (steam_user_LydiaMorley):
  - steam_item_273110, Title: Counter-Strike Nexon: Zombies
  - steam_item_730, Title: Counter-Strike: Global Offensive
  - steam_item_440, Title: Team Fortress 2
Top 10 recommendations for user 'steam_user_LydiaMorley':
  - [movie] Item: movielens_item_2731, Title: 400 Blows, The (Les quatre cents coups) (1959) , Score: 0.4314
  - [movie] Item: movielens_item_7210, Title: My Darling Clementine (1946) , Score: 0.4183
  - [movie] Item: movielens_item_4037, Title: House of Games (1987) , Score: 0.4057
  - [movie] Item: movielens_item_3068, Title: Verdict, The (1982) , Score: 0.4037
  - [movie] Item: movielens_item_6016, Title: City of God (Cidade de Deus) (2002) , Score: 0.3927
  - [movie] Item: movielens_item_3486, Title: Devil Girl From Mars (1954) , Score: 0.3836
  - [movie] Item: movielens_item_5451, Title: Pumpkin (2002) , Score: 0.3813
  - [movie] Item: movielens_item_51418, Title: Breaking and Entering (2006) , Score: 0.3778
  - [m

In [ ]:
steam_interactions.tail()


,user_id,item_id,rating
52468,steam_user_76561198312638244,steam_item_70,1.0
52469,steam_user_76561198312638244,steam_item_362890,1.0
52470,steam_user_LydiaMorley,steam_item_273110,1.0
52471,steam_user_LydiaMorley,steam_item_730,1.0
52472,steam_user_LydiaMorley,steam_item_440,1.0
